## Smart Document Search Engine
---

Goal: Build a system that understands users query and returns the relevant documents (Basic version of Google Search engine)

In [1]:
# 1. Data processing

corpus = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data",
    "Deep learning uses neural networks with multiple layers to recognize complex patterns",
    "Natural language processing helps computers understand and generate human language",
    "Computer vision allows machines to interpret visual information from images and videos",
    "Reinforcement learning trains agents to make decisions through rewards and penalties",
    "Artificial intelligence is transforming industries from healthcare to finance",
    "Neural networks are inspired by the human brain structure and function",
    "Supervised learning requires labeled data to train predictive models",
    "Unsupervised learning finds hidden patterns in unlabeled datasets",
    "Transfer learning allows models to reuse knowledge from one task to another",
    "Convolutional neural networks excel at image recognition and classification tasks",
    "Recurrent neural networks are designed for sequential data like text and time series",
    "Generative adversarial networks create realistic synthetic data through competition",
    "Attention mechanisms help models focus on relevant parts of the input",
    "Transformers have revolutionized natural language processing with self-attention",
    "Large language models can generate human-like text and answer questions",
    "Ethical AI focuses on building fair transparent and accountable systems",
    "Data preprocessing is crucial for building accurate machine learning models",
    "Feature engineering improves model performance by creating meaningful input variables",
    "Model evaluation metrics like accuracy precision and recall measure performance"
]

print(f"Total documents: {len(corpus)}")
print(f"Average document length: {sum(len(doc.split()) for doc in corpus) / len(corpus):.4f} words")

for i in [0, 5, 10, 15]:
    print(f"\n{i}: {corpus[i]}")

Total documents: 20
Average document length: 10.5000 words

0: Machine learning is a subset of artificial intelligence that enables systems to learn from data

5: Artificial intelligence is transforming industries from healthcare to finance

10: Convolutional neural networks excel at image recognition and classification tasks

15: Large language models can generate human-like text and answer questions


In [2]:
import re

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

processed_corpus = [preprocess_text(text) for text in corpus]

print(f"Original: {corpus[0]}")
print(f"Processed: {processed_corpus[0]}")

Original: Machine learning is a subset of artificial intelligence that enables systems to learn from data
Processed: machine learning is a subset of artificial intelligence that enables systems to learn from data


In [5]:
from collections import Counter

def train_bpe(corpus, vocab_size):
    corpus_dict = {}
    for sent in corpus:
        sent = sent.split()
        for word in sent:
            if word in corpus_dict:
                corpus_dict[tuple(word)] += 1
            else:
                corpus_dict[tuple(word)] = 1
    
    vocab = set(char for char_tuple in corpus_dict.keys() for char in char_tuple)
    merges = []

    while len(vocab) < vocab_size:
        pair_counts = {}
        for char_tuple, freq in corpus_dict.items():
            for i in range(len(char_tuple) - 1):
                pair = (char_tuple[i], char_tuple[i+1])

                if pair in pair_counts:
                    pair_counts[pair] += freq
                else:
                    pair_counts[pair] = freq
            
        if not pair_counts:
                break

        best_pair = max(pair_counts, key=pair_counts.get)
        token = best_pair[0] + best_pair[1]
        merges.append(best_pair)

        new_corpus_dict = {}

        for word, freq in corpus_dict.items():
            word_list = list(word)
            i = 0

            while i < len(word_list) - 1:
                if word_list[i] + word_list[i+1] == token:
                    word_list[i:i+2] = [token]
                else:
                    i+=1

            new_corpus_dict[tuple(word_list)] = freq

        corpus_dict = new_corpus_dict
        vocab.add(token)

        # print(f"Merged: {token} | Vocab size: {len(vocab)}")

    # print(vocab, corpus_dict, sep="\n")
    return vocab, merges


bpe_vocab, bpe_merges = train_bpe(processed_corpus, vocab_size=200)
print(f"BPE Vocabulary size: {len(bpe_vocab)}")
print(f"Sample tokens: {list(bpe_vocab)[:20]}")

token_to_id = {
    '<PAD>': 0,
    '<UNK>': 1
}

idx = 2
for token in bpe_vocab:
    if token not in token_to_id:
        token_to_id[token] = idx
        idx += 1

id_to_token = {
    id: token
    for token, id in token_to_id.items()
}

print(f"Total vocab size (with special tokens): {len(token_to_id)}")

BPE Vocabulary size: 200
Sample tokens: ['a', 'te', 'er', 'ut', 'ed', 'transfor', 'accur', 'ire', 'recogn', 'ima', 'model', 'proc', 'tic', 'gen', 'pro', 'en', 'ing', 'are', 'ma', 'learn']
Total vocab size (with special tokens): 202


In [4]:
def tokenize_with_bpe(text, merges):
    tokens = list(text)
    
    for merge in merges:
        merged_token = merge[0] + merge[1]  
        
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == merge[0] and tokens[i+1] == merge[1]:
                new_tokens.append(merged_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        
        tokens = new_tokens
    
    return tokens

def tokens_to_ids(tokens, token_to_id):
    ids = []
    for token in tokens:
        token_id = token_to_id.get(token, token_to_id['<UNK>'])
        ids.append(token_id)
    return ids

def encode_document(text, merges, token_to_id):
    words = text.split()
    all_ids = []
    
    for word in words:
        tokens = tokenize_with_bpe(word, merges)
        ids = tokens_to_ids(tokens, token_to_id)
        all_ids.extend(ids)
    
    return all_ids

doc_text = processed_corpus[0]
doc_ids = encode_document(doc_text, bpe_merges, token_to_id)

print(f"Original text: {doc_text}")
print(f"Document IDs: {doc_ids[:10]}")
print(f"Decoded tokens: {[id_to_token[id] for id in doc_ids[:10]]}")

Original text: machine learning is a subset of artificial intelligence that enables systems to learn from data
Document IDs: [68, 43, 74, 2, 54, 195, 108, 23, 173, 196]
Decoded tokens: ['machine', 'learning', 'is', 'a', 'subset', 'of', 'artificial', 'intelligence', 'that', 'enables']


In [6]:
# 2. TD-IDF vectorization

word_vocab = {}
idx = 0

for doc in processed_corpus:
    words = doc.split()
    for word in words:
        if word not in word_vocab:
            word_vocab[word] = idx
            idx += 1

print(f"Word vocabulary size: {len(word_vocab)}")
print(f"Sample words: {list(word_vocab.keys())[:10]}")

Word vocabulary size: 142
Sample words: ['machine', 'learning', 'is', 'a', 'subset', 'of', 'artificial', 'intelligence', 'that', 'enables']


In [15]:
import torch

num_docs = len(processed_corpus)
vocab_size = len(word_vocab)

tf_matrix = torch.zeros((num_docs, vocab_size), dtype=torch.float32)

for i, doc in enumerate(processed_corpus):
    words = doc.split()
    for word in words:
        j = word_vocab[word]
        tf_matrix[i, j] += 1

row_sums = tf_matrix.sum(dim=1, keepdim=True)
tf_matrix_normalized = tf_matrix/row_sums

print(f"TF Matrix shape: {tf_matrix_normalized.shape}")
print(f"Sample (doc 0, first 5 words): {tf_matrix_normalized[0, :5]}")

df_vector = tf_matrix.sum(dim=0)
N = torch.tensor(num_docs, dtype=torch.float32)

idf_vector = torch.log(N / df_vector)
print(f"IDF Vector shape: {idf_vector.shape}")
print(f"Sample IDF values: {idf_vector[:5]}")

tf_idf_matrix = tf_matrix_norm * idf_vector
print(f"TF-IDF Matrix shape: {tf_idf_matrix.shape}")
print(f"Sample (doc 0, first 5 words): {tf_idf_matrix[0, :5]}")

def encode_query(query, word_vocab, tf_idf_matrix):
    query = preprocess_text(query)
    words = query.split()

    query_tf = torch.zeros(len(word_vocab), dtype=torch.float32)
    for word in words:
        if word in word_vocab:
            j = word_vocab[word]
            query_tf[j] += 1

    query_sum = query_tf.sum()
    if query_sum > 0:
        query_tf = query_tf / query_sum

    query_tfidf = query_tf * idf_vector

    return query_tfidf

query = "neural networks deep learning"
query_vector = encode_query(query, word_vocab, tf_idf_matrix)
print(f"Query vector shape: {query_vector.shape}")

TF Matrix shape: torch.Size([20, 142])
Sample (doc 0, first 5 words): tensor([0.0667, 0.0667, 0.0667, 0.0667, 0.0667])
IDF Vector shape: torch.Size([142])
Sample IDF values: tensor([2.3026, 1.0498, 1.8971, 2.9957, 2.9957])
TF-IDF Matrix shape: torch.Size([20, 142])
Sample (doc 0, first 5 words): tensor([0.1535, 0.0700, 0.1265, 0.1997, 0.1997])
Query vector shape: torch.Size([142])


In [16]:
# 3. Search and similarity
def cosine_similarity(vec1, vec2):
    dot_product = torch.dot(vec1, vec2)
    norm1 = torch.norm(vec1)
    norm2 = torch.norm(vec2)
    
    if norm1 == 0 or norm2 == 0:
        return torch.tensor(0.0)
    
    return dot_product / (norm1 * norm2)

def search(query, word_vocab, tf_idf_matrix, idf_vector, corpus, top_k=3):
    
    query_vector = encode_query(query, word_vocab, tf_idf_matrix)
    
    similarities = []
    for i in range(len(corpus)):
        doc_vector = tf_idf_matrix[i]
        sim = cosine_similarity(query_vector, doc_vector)
        similarities.append((i, sim.item()))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    top_results = similarities[:top_k]
    
    print(f"\n{'='*60}")
    print(f"Query: '{query}'")
    print(f"{'='*60}")
    
    for rank, (doc_idx, score) in enumerate(top_results, 1):
        print(f"\nRank {rank} | Score: {score:.4f}")
        print(f"Document: {corpus[doc_idx]}")
    
    return top_results

In [19]:
# Test Query 1: Neural Networks related
search("neural networks deep learning", word_vocab, tf_idf_matrix, idf_vector, corpus, top_k=3)

# Test Query 2: NLP related
search("natural language processing text", word_vocab, tf_idf_matrix, idf_vector, corpus, top_k=3)

# Test Query 3: Computer Vision related
search("image recognition visual", word_vocab, tf_idf_matrix, idf_vector, corpus, top_k=3)

# Test Query 4: Ethics related
search("ethical fair transparent AI", word_vocab, tf_idf_matrix, idf_vector, corpus, top_k=3)

# Test Query 5: Model evaluation related
search("model evaluation accuracy precision", word_vocab, tf_idf_matrix, idf_vector, corpus, top_k=3)


Query: 'neural networks deep learning'

Rank 1 | Score: 0.4536
Document: Deep learning uses neural networks with multiple layers to recognize complex patterns

Rank 2 | Score: 0.1497
Document: Neural networks are inspired by the human brain structure and function

Rank 3 | Score: 0.1433
Document: Convolutional neural networks excel at image recognition and classification tasks

Query: 'natural language processing text'

Rank 1 | Score: 0.4773
Document: Natural language processing helps computers understand and generate human language

Rank 2 | Score: 0.4160
Document: Transformers have revolutionized natural language processing with self-attention

Rank 3 | Score: 0.2357
Document: Large language models can generate human-like text and answer questions

Query: 'image recognition visual'

Rank 1 | Score: 0.4196
Document: Convolutional neural networks excel at image recognition and classification tasks

Rank 2 | Score: 0.1920
Document: Computer vision allows machines to interpret visual i

[(19, 0.6766165494918823), (18, 0.1078069880604744), (0, 0.0)]